# RAG Evaluation Notebook

Evaluates the hybrid RAG system (vector + BM25 + graph) on regulatory compliance queries.

## Metrics
- **Hit Rate @5**: Proportion of queries where the relevant chunk appears in top-5 results
- **MRR (Mean Reciprocal Rank)**: Average of 1/rank_of_first_relevant across queries
- **NDCG@5**: Normalized Discounted Cumulative Gain at rank 5
- **Faithfulness**: LLM-as-judge check that cited regulations come from retrieved chunks
- **Answer Relevance**: Cosine similarity between answer embedding and question embedding

In [ ]:
import os
import sys

sys.path.insert(0, '..')

os.environ['DJANGO_SETTINGS_MODULE'] = 'config.settings.local'
os.environ['USE_MOCK_BQ'] = 'true'
os.environ['USE_MOCK_GCS'] = 'true'

from compliance_agent.bootstrap import bootstrap_django

bootstrap_django()

In [ ]:
from pathlib import Path

# Discover the 5 fixture regulatory documents
fixture_dir = Path("../tests/fixtures/regulatory_docs")
docs = sorted(fixture_dir.glob("*.txt"))
print(f"Found {len(docs)} regulatory documents:")
for d in docs:
    print(f"  {d.name}")

In [ ]:
# Test queries with ground-truth relevant documents
TEST_QUERIES = [
    {
        'query': 'When must a PEP alert be escalated to human review?',
        'relevant_docs': ['uiaf_resolucion_285_2007', 'cnbv_cfpior_art95', 'sbs_resolucion_2019_1874'],
        'relevant_articles': ['Art. 14', 'Art. 95', 'Art. 17'],
    },
    {
        'query': '¿Cuál es el umbral de reporte en efectivo para Colombia?',
        'relevant_docs': ['uiaf_resolucion_285_2007', 'uiaf_circular_002_2020'],
        'relevant_articles': ['Art. 7', 'Art. 1'],
    },
    {
        'query': 'Plazos para reportar operaciones sospechosas en México',
        'relevant_docs': ['cnbv_cfpior_art95'],
        'relevant_articles': ['Art. 97'],
    },
    {
        'query': 'Record retention requirements for suspicious transaction reports in Peru',
        'relevant_docs': ['sbs_resolucion_2019_1874'],
        'relevant_articles': ['Art. 23'],
    },
    {
        'query': 'How to classify customers by risk level for AML compliance',
        'relevant_docs': ['uiaf_circular_002_2020', 'sbs_circular_g_2140_2006'],
        'relevant_articles': ['Art. 5', 'Art. 1'],
    },
]

print(f'Test set: {len(TEST_QUERIES)} queries')

In [ ]:
import math


def hit_rate_at_k(results: list[dict], relevant_docs: list[str], k: int = 5) -> float:
    """HR@k = 1 if any relevant doc in top-k, else 0"""
    top_k_refs = [r['document_ref'] for r in results[:k]]
    return 1.0 if any(rel in top_k_refs for rel in relevant_docs) else 0.0

def reciprocal_rank(results: list[dict], relevant_docs: list[str]) -> float:
    """1/rank of first relevant document"""
    for rank, result in enumerate(results, 1):
        if result['document_ref'] in relevant_docs:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(results: list[dict], relevant_docs: list[str], k: int = 5) -> float:
    """NDCG@k = DCG@k / IDCG@k"""
    def dcg(ranked):
        return sum(
            (1.0 if r['document_ref'] in relevant_docs else 0.0) / math.log2(i + 2)
            for i, r in enumerate(ranked[:k])
        )
    ideal = sorted([1.0] * min(len(relevant_docs), k) + [0.0] * k, reverse=True)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal[:k]))
    return dcg(results) / idcg if idcg > 0 else 0.0

print('Metric functions defined.')

In [ ]:
from compliance_agent.rag import HFEmbedder
from compliance_agent.rag.graph_retriever import GraphRetriever

# Build shared embedder (used by both hybrid retriever and graph retriever below)
embedder = HFEmbedder()

# GraphRetriever constants sanity check
print(f"SEED_WEIGHT    : {GraphRetriever.SEED_WEIGHT}")
print(f"NEIGHBOR_WEIGHT: {GraphRetriever.NEIGHBOR_WEIGHT}")
assert GraphRetriever.SEED_WEIGHT > GraphRetriever.NEIGHBOR_WEIGHT, "Seed documents must always outscore graph-expanded neighbors"  # noqa: S101

graph_retriever = GraphRetriever(embedder=embedder, seed_top_k=3)
print("\nGraphRetriever instantiated successfully.")
print("To run full graph evaluation, connect to a DB with indexed documents and related_articles populated.")

## Graph Retrieval Layer

`GraphRetriever` wraps `VectorStoreRetriever` and expands seed results via the `related_articles` M2M graph (1-hop traversal). Seed documents receive `SEED_WEIGHT=1.0 × score`; graph-expanded neighbors receive `NEIGHBOR_WEIGHT=0.5`. This improves precision for multi-article regulatory reasoning (e.g., PEP rules that span UIAF Art.14 → CNBV Art.95 → SBS Art.17).

In [ ]:
import asyncio

import pandas as pd

from compliance_agent.rag import (
    HybridRetriever,
    SparseVectorRetriever,
    VectorStoreRetriever,
)
from compliance_agent.rag.sparse_vectorizer import SparseVectorizer

# embedder is defined in the cell above (cell-5)
sparse_vectorizer = SparseVectorizer()
vector_retriever = VectorStoreRetriever(embedder)
sparse_retriever = SparseVectorRetriever(sparse_vectorizer)
hybrid_retriever = HybridRetriever(
    vector_retriever=vector_retriever,
    sparse_retriever=sparse_retriever,
)

ground_truth = {q['query']: q['relevant_docs'] for q in TEST_QUERIES}


async def run_all_queries():
    results = []
    for test in TEST_QUERIES:
        query = test['query']
        relevant = test['relevant_docs']
        try:
            chunks = await hybrid_retriever.retrieve(query, top_k=5)
            retrieved = [{'document_ref': c.document_ref, 'score': c.score} for c in chunks]
        except Exception:
            # No DB connection in notebook — return empty for metric demo
            retrieved = []

        hr = hit_rate_at_k(retrieved, relevant, k=5)
        mrr_val = reciprocal_rank(retrieved, relevant)
        ndcg_val = ndcg_at_k(retrieved, relevant, k=5)
        results.append({
            'query': query[:60],
            'hit_rate@5': hr,
            'mrr': mrr_val,
            'ndcg@5': ndcg_val,
        })
    return results


results = asyncio.run(run_all_queries())
df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nMean Hit Rate@5 : {df['hit_rate@5'].mean():.3f}")
print(f"Mean MRR        : {df['mrr'].mean():.3f}")
print(f"Mean NDCG@5     : {df['ndcg@5'].mean():.3f}")